# 03 — Error analysis on out-of-fold predictions
Run the pipeline first so `cache/oof_train.pkl` exists:

`python src/pipeline.py run --data dataset --no-test --frac 0.3`

Most leaderboard gains come from reading these errors and turning patterns into features or rules.

In [ ]:
# --- setup: run from the repo root or from notebooks/ ---
import sys, os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "dataset"                 # real data: dataset/train, dataset/test
if not (DATA / "train").exists():       # fall back to the synthetic dev data
    DATA = ROOT / "fake_dataset"
CACHE = ROOT / "cache"
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("data:", DATA)

In [ ]:
from metric import macro_f05, breakdown, f05
from select_matches import one_to_one, decode_threshold
from pipeline import decode, to_ids
art = pickle.load(open(CACHE / "oof_train.pkl", "rb"))
X, s1, s23, truth, cfg = art["X"], art["s1"], art["s23"], art["truth"], art["cfg"]
X["s1_id"] = s1["entity_id"].values[X.i1]; X["s23_id"] = s23["entity_id"].values[X.i2]
ids = s1["entity_id"].tolist()
pred = to_ids(decode(X[["i1", "i2", "p"]], cfg), s1, s23)
print("decoder:", cfg)
print(breakdown(pred, truth, ids))
country = dict(zip(s1.entity_id, s1.country))
per = pd.DataFrame({"id": ids, "f": [f05(pred.get(i, set()), truth.get(i, set())) for i in ids],
                    "country": [country[i] for i in ids], "n_true": [len(truth.get(i, ())) for i in ids]})
display(per.groupby("country")["f"].agg(["mean", "size"]))
display(per.groupby(per.n_true.clip(upper=4))["f"].agg(["mean", "size"]).rename_axis("#true matches (4=4+)"))

## Score vs threshold — how flat is the optimum? (flat = robust on private LB)

In [ ]:
ts = np.linspace(0.1, 0.95, 35)
res = []
for t in ts:
    for o2o in (False, True):
        d = one_to_one(X[["i1", "i2", "p"]]) if o2o else X[["i1", "i2", "p"]]
        pr = to_ids(decode_threshold(d, t), s1, s23)
        b = breakdown(pr, truth, ids); res.append(dict(t=t, o2o=o2o, **b))
res = pd.DataFrame(res)
for o2o, g in res.groupby("o2o"):
    plt.plot(g.t, g["all"], label=f"all (one_to_one={o2o})")
g = res[res.o2o]
plt.plot(g.t, g.singletons, "--", label="singletons"); plt.plot(g.t, g.with_matches, "--", label="with matches")
plt.xlabel("threshold"); plt.ylabel("F0.5"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Calibration (the expected-F decoder assumes calibrated probabilities)

In [ ]:
bins = np.linspace(0, 1, 11)
b = np.clip(np.digitize(X.p, bins) - 1, 0, 9)
cal = X.groupby(b).agg(pred=("p", "mean"), actual=("y", "mean"), n=("y", "size"))
plt.plot([0, 1], [0, 1], "k:"); plt.plot(cal.pred, cal.actual, marker="o")
plt.xlabel("predicted p"); plt.ylabel("observed match rate"); plt.show(); cal

## False positives: confident predictions that are wrong (these cost the most under F0.5)

In [ ]:
S1, S23 = s1, s23
def side_by_side(rows):
    return pd.DataFrame(dict(p=rows.p.round(3), y=rows.y,
        s1_name=S1.business_name.values[rows.i1], s23_name=S23.business_name.values[rows.i2],
        s1_addr=S1.business_address.values[rows.i1], s23_addr=S23.business_address.values[rows.i2],
        country=S1.country.values[rows.i1]))
fp = X[(X.y == 0)].sort_values("p", ascending=False).head(30)
side_by_side(fp)

## False negatives: true matches the model scored low

In [ ]:
fn = X[(X.y == 1)].sort_values("p").head(30)
side_by_side(fn)

## Singletons we wrongly matched (each one costs a full 1.0)

In [ ]:
wrong_single = [i for i in ids if not truth.get(i) and pred.get(i)]
print(len(wrong_single), "singletons given matches")
rows = X[X.s1_id.isin(wrong_single[:40])].sort_values(["i1", "p"], ascending=[True, False]).groupby("i1").head(1)
side_by_side(rows)

## Error patterns by feature (where do errors concentrate?)

In [ ]:
hi = X[X.p >= cfg.get("t", 0.5)]
for col in ["pcode", "num_conf", "core_eq", "country_eq", "src3"]:
    if col in X:
        display(hi.groupby(col)["y"].agg(precision="mean", n="size").T.rename_axis(col))

## Feature importance (permutation, on a held-out group split)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit
c2 = art["c2"]
tr, va = next(GroupShuffleSplit(1, test_size=0.25, random_state=0).split(X, X.y, X.i1))
m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08).fit(X.iloc[tr][c2], X.y.iloc[tr])
va = va[:50_000]
pi = permutation_importance(m, X.iloc[va][c2], X.y.iloc[va], n_repeats=3, random_state=0, scoring="average_precision")
imp = pd.Series(pi.importances_mean, index=c2).sort_values(ascending=False)
imp.head(25).iloc[::-1].plot.barh(figsize=(6, 7)); plt.title("permutation importance (AP drop)"); plt.show()
print("near-zero importance (candidates to drop):", list(imp[imp.abs() < 1e-4].index))